In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from utils.data_loading import load_data as load_v1_original, load_test_data
from utils.data_loadingV2 import load_data as load_v2_temporal
from utils.data_loadingV3_random import load_data as load_v3_random
from utils.data_loadingV4_file import load_data as load_v4_file
from utils.ranksim_loss import RankSimWithBaseLoss
from models.vanilla_rnn import VanillaRNN

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [3]:
CONFIG = {
    'TRAIN_PATH': PROJECT_ROOT / "datasets_challenge" / "train",
    'VAL_RATIO': 0.2,
    'BATCH_SIZE': 64,
    'COLUMNS_TO_DROP': ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': 'vxCG',
    'TARGET': 'sideSlip',
    'SCALING': True,
    'SEED': 42,
    'NUM_BINS': 100,
    'DIST_CSV': PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv",
}

In [4]:
print("Loading V1: Original Split")
train_v1, val_v1, pipeline_v1 = load_v1_original(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=0.8,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V2: Temporal Split")
train_v2, val_v2, scaler_v2, _, _ = load_v2_temporal(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.15,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V3: Random Split")
train_v3, val_v3, scaler_v3, _, _ = load_v3_random(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.2,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V4: File-Based Split (File 7)")
train_v4, val_v4, scaler_v4, _, _ = load_v4_file(
    train_path=CONFIG['TRAIN_PATH'],
    val_file_num=11,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

def extract_targets(loader):
    targets = []
    for _, y in loader:
        targets.append(y)
    return torch.cat(targets).numpy().flatten()

train_targets_v1 = extract_targets(train_v1)
val_targets_v1 = extract_targets(val_v1)

train_targets_v2 = extract_targets(train_v2)
val_targets_v2 = extract_targets(val_v2)

train_targets_v3 = extract_targets(train_v3)
val_targets_v3 = extract_targets(val_v3)

train_targets_v4 = extract_targets(train_v4)
val_targets_v4 = extract_targets(val_v4)

print(f"\nDataset Sizes:")
print(f"V1 Original: Train {len(train_targets_v1):,} | Val {len(val_targets_v1):,}")
print(f"V2 Temporal: Train {len(train_targets_v2):,} | Val {len(val_targets_v2):,}")
print(f"V3 Random:   Train {len(train_targets_v3):,} | Val {len(val_targets_v3):,}")
print(f"V4 File:     Train {len(train_targets_v4):,} | Val {len(val_targets_v4):,}")

Loading V1: Original Split
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\trai

In [9]:
class VanillaRNN(nn.Module):
    """
    Vanilla RNN that exposes penultimate features
    (required for RankSim / FDS)
    """
    def __init__(self, input_size=22, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            nonlinearity='relu',
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, return_features=False):
        _, h = self.rnn(x)
        feats = self.dropout(h[-1])  # (B, hidden_size)

        if return_features:
            return feats

        return self.regressor(feats)


In [13]:
def train_eval_with_ranksim(
    train_loader,
    val_loader,
    epochs=30,
    lambda_rank=0.1
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model
    model = VanillaRNN(
        input_size=22,
        hidden_size=64,
        num_layers=1,
        dropout=0.0
    ).to(device)

    # Base regression loss
    criterion = RankSimWithBaseLoss(
    base_loss='l1',
    temperature=0.1,
    lambda_ranksim=lambda_rank   # start small
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    # =======================
    # Training
    # =======================
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).float()

            optimizer.zero_grad()

            feats = model(x, return_features=True)
            preds = model.regressor(feats).squeeze(-1)

            loss = criterion(preds, feats, y)

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(
            f"Epoch {epoch:02d} | "
            f"Total Loss: {epoch_loss / len(train_loader):.4f}"
        )

    # =======================
    # Validation (MAE)
    # =======================
    model.eval()
    abs_errors = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device).float()

            feats = model(x, return_features=True)
            preds = model.regressor(feats).squeeze(-1)

            abs_errors.append(torch.abs(preds - y))

    val_mae = torch.cat(abs_errors).mean().item()

    return model, val_mae


In [14]:
trained_models = {}
ranksim_results = {}

print("\n LDS on V1 (Original split)")
trained_models['V1'], ranksim_results['V1'] = train_eval_with_ranksim(
    train_v1, val_v1, epochs=30, lambda_rank=0.05
)

print("\n LDS on V2 (Temporal split)")
trained_models['V2'], ranksim_results['V2'] = train_eval_with_ranksim(
    train_v2, val_v2, epochs=30, lambda_rank=0.05
)

print("\n LDS on V3 (Random split)")
trained_models['V3'], ranksim_results['V3'] = train_eval_with_ranksim(
    train_v3, val_v3, epochs=30, lambda_rank=0.05
)

print("\n LDS on V4 (File-based split)")
trained_models['V4'], ranksim_results['V4'] = train_eval_with_ranksim(
    train_v4, val_v4, epochs=30, lambda_rank=0.05
)



 LDS on V1 (Original split)
Epoch 00 | Total Loss: 0.3173
Epoch 01 | Total Loss: 0.2251
Epoch 02 | Total Loss: 0.2061
Epoch 03 | Total Loss: 0.1950
Epoch 04 | Total Loss: 0.1880
Epoch 05 | Total Loss: 0.1826
Epoch 06 | Total Loss: 0.1787
Epoch 07 | Total Loss: 0.1757
Epoch 08 | Total Loss: 0.1734
Epoch 09 | Total Loss: 0.1709
Epoch 10 | Total Loss: 0.1686
Epoch 11 | Total Loss: 0.1669
Epoch 12 | Total Loss: 0.1651
Epoch 13 | Total Loss: 0.1635
Epoch 14 | Total Loss: 0.1617
Epoch 15 | Total Loss: 0.1604
Epoch 16 | Total Loss: 0.1591
Epoch 17 | Total Loss: 0.1576
Epoch 18 | Total Loss: 0.1563
Epoch 19 | Total Loss: 0.1554
Epoch 20 | Total Loss: 0.1543
Epoch 21 | Total Loss: 0.1533
Epoch 22 | Total Loss: 0.1523
Epoch 23 | Total Loss: 0.1514
Epoch 24 | Total Loss: 0.1504
Epoch 25 | Total Loss: 0.1499
Epoch 26 | Total Loss: 0.1490
Epoch 27 | Total Loss: 0.1484
Epoch 28 | Total Loss: 0.1477
Epoch 29 | Total Loss: 0.1466

 LDS on V2 (Temporal split)
Epoch 00 | Total Loss: 0.3209
Epoch 01 | T

In [15]:
print("\nFinal Comparison (Validation MAE)")
for k, v in ranksim_results.items():
    print(f"{k}: MAE = {v:.4f}")



Final Comparison (Validation MAE)
V1: MAE = 0.1333
V2: MAE = 0.1560
V3: MAE = 0.2549
V4: MAE = 0.1671
